In [2]:
import os
import sys
from pathlib import Path
print(os.getcwd())

ROOT_DIR = Path(os.getcwd()).parent.parent
print(ROOT_DIR)
sys.path.insert(0, str(ROOT_DIR))

/home/tinhanhnguyen/Desktop/HK7/Capstone/CAPSTONE_PROJECT/agentic_ai/test
/home/tinhanhnguyen/Desktop/HK7/Capstone/CAPSTONE_PROJECT


In [3]:
from agentic_ai.tools.clients import *
from agentic_ai.tools.schema.artifact import ImageObjectInterface, SegmentObjectInterface, VideoInterface
from agentic_ai.tools.type import *

In [4]:
# CONSTANT
MILVUS_HOST = "localhost"
MILVUS_PORT = 19530
MILVUS_URI = f"http://{MILVUS_HOST}:{MILVUS_PORT}"

IMAGE_COLLECTION_NAME = "image_milvus"
IMAGE_VISUAL_PARAM = {
    "type_config": "dense",
    "dimension": 512,
    "metric_type": "COSINE",
    "index_type": "HNSW",
    "description": "Image embeddings for video frames",
    "index_params": {"M": 64, "efConstruction": 100},
}
IMAGE_CAPTION_DENSE_PARAM = {
    "type_config": "dense",
    "dimension": 768,
    "metric_type": "COSINE",
    "index_type": "HNSW",
    "description": "Text embeddings for image captions",
    "index_params": {"M": 64, "efConstruction": 100},
}
IMAGE_CAPTION_SPARSE_PARAM = {
    "type_config": "sparse",
    "dimension": 1000000,  # ignored at runtime
    "metric_type": "BM25",
    "index_type": "SPARSE_INVERTED_INDEX",
    "description": "BM25 sparse index for image captions",
    "index_params": {"inverted_index_algo": "DAAT_MAXSCORE"},
}
IMAGE_VISUAL_FIELD = "visual_embedding_field"
IMAGE_CAPTION_FIELD = "caption_embedding_field"
IMAGE_SPARSE_FIELD = "caption_sparse_embedding_field"


SEGMENT_CAPTION_COLLECTION_NAME = "segment_milvus"
SEGMENT_CAPTION_DENSE_PARAM = {
    "type_config": "dense",
    "dimension": 768,
    "metric_type": "COSINE",
    "index_type": "HNSW",
    "description": "Text embeddings for segment captions",
    "index_params": {"M": 64, "efConstruction": 100},
}
SEGMENT_CAPTION_SPARSE_PARAM = {
    "type_config": "sparse",
    "dimension": 1000000,  # ignored at runtime
    "metric_type": "BM25",
    "index_type": "SPARSE_INVERTED_INDEX",
    "description": "BM25 sparse index for segment captions",
    "index_params": {"inverted_index_algo": "DAAT_MAXSCORE"},
}

SEGMENT_DENSE_FIELD = "caption_embedding_field"
SEGMENT_SPARSE_FIELD = "caption_sparse_embedding_field"



In [5]:
image_milvus_client = ImageMilvusClient(
    uri=MILVUS_URI,
    collection_name=IMAGE_COLLECTION_NAME,
    visual_param=IMAGE_VISUAL_PARAM,
    caption_param=IMAGE_CAPTION_DENSE_PARAM,
    sparse_param=IMAGE_CAPTION_SPARSE_PARAM,
    visual_ann_field=IMAGE_VISUAL_FIELD,
    caption_ann_field=IMAGE_CAPTION_FIELD,
    sparse_field=IMAGE_SPARSE_FIELD,
)

segment_caption_client = SegmentCaptionImageMilvusClient(
    uri=MILVUS_URI,
    collection_name=SEGMENT_CAPTION_COLLECTION_NAME,
    dense_param=SEGMENT_CAPTION_DENSE_PARAM,
    sparse_param=SEGMENT_CAPTION_SPARSE_PARAM,
    dense_field=SEGMENT_DENSE_FIELD,
    sparse_field=SEGMENT_SPARSE_FIELD,
)

await image_milvus_client.connect()
await segment_caption_client.connect()

In [6]:
# await image_milvus_client.close()
# await segment_caption_client.close()

In [7]:
image_emebedding_settings = ImageEmbeddingSettings(
    model_name='open_clip',
    device='cuda',
    batch_size=32
)
text_emebedding_settings = TextEmbeddingSettings(
    model_name='mmbert',
    device='cuda',
    batch_size=32
)

img_txt_client = ImageEmbeddingClient(base_url='http://localhost:8003')
txt_client = TextEmbeddingClient(base_url='http://localhost:8005')


external_client = ExternalEncodeClient(img_text_client=img_txt_client, img_text_settings=image_emebedding_settings, txt_settings=text_emebedding_settings, txt_client=txt_client)

await external_client.connect()

2025-11-04 10:48:23.942 | INFO     | agentic_ai.tools.clients.external.base:load_model:126 - service-image-embedding_model_loaded
2025-11-04 10:48:23.946 | INFO     | agentic_ai.tools.clients.external.base:load_model:126 - service-text-embedding_model_loaded


In [8]:
MINIO_HOST = "localhost"       # use localhost when running locally
MINIO_PORT = 9000
MINIO_USER = "minioadmin"
MINIO_PASSWORD = "minioadmin"
MINIO_ACCESS_KEY = "minioadmin"
MINIO_SECRET_KEY = "minioadmin"
MINIO_SECURE = False  
MINIO_ENDPOINT = f"{MINIO_HOST}:{MINIO_PORT}"

storage_client = StorageClient(
    host=MINIO_HOST,
    port=MINIO_PORT,#type:ignore
    access_key=MINIO_ACCESS_KEY,
    secret_key=MINIO_SECRET_KEY,
    secure=MINIO_SECURE,
)

storage_client._ensure_bucket('anonymous')

In [9]:
from sqlalchemy import text

POSTGRE_USER = "prefect"
POSTGRE_PASSWORD = "prefect"
POSTGRE_HOST = "localhost"       # use localhost for local setup
POSTGRE_PORT = 5432
POSTGRE_DB = "prefect"

POSTGRE_DATABASE_URL = (
    f"postgresql+asyncpg://{POSTGRE_USER}:{POSTGRE_PASSWORD}"
    f"@{POSTGRE_HOST}:{POSTGRE_PORT}/{POSTGRE_DB}"
)
postgres_client = PostgresClient(
    database_url=POSTGRE_DATABASE_URL
)
async with postgres_client.get_session() as session:
    result = await session.execute(text("SELECT version();"))
    version = result.scalar_one()
    print(f"🗄️ PostgreSQL connection successful.\n   Version: {version}")

🗄️ PostgreSQL connection successful.
   Version: PostgreSQL 14.19 (Debian 14.19-1.pgdg13+1) on x86_64-pc-linux-gnu, compiled by gcc (Debian 14.2.0-19) 14.2.0, 64-bit


In [10]:

from llama_index.llms.google_genai import GoogleGenAI
from dotenv import load_dotenv
load_dotenv()


llm = GoogleGenAI(
    model='gemini-2.5-flash',
        
)

tool_factory = ToolFactory(
    image_milvus_client=image_milvus_client,
    segment_milvus_client=segment_caption_client,
    postgres_client=postgres_client,
    minio_client=storage_client,
    external_client=external_client,
    llm_as_tools=llm
)

In [11]:
tools = tool_factory.get_all_tools()

get_video_from_segment
get_video_from_image
get_asr_from_video
get_all_segment_info_from_video_interface
get_segments
get_images
extract_frames_by_time_window
extract_frame_time
frame_to_timecode
timecode_to_frame
from_index_to_time
from_range_index_to_range_time
from_time_to_index
from_range_time_to_range_index
read_image
read_segment
get_related_asr_from_image
get_related_asr_from_segment
get_images_from_visual_query
get_images_from_caption_query
get_segments_from_event_query
get_images_from_multimodal_query
find_similar_images_from_image
enhance_visual_query
enhance_textual_query
caption_new_image


In [12]:
LIST_VIDEO_IDS = ['video1_111', 'video2_222', 'video3_333']
USER_ID = 'anonymous'

In [13]:
# testing search tools

VISUAL_QUERY = "traffic police officer in beige uniform and red cap sitting at a roadside table with a man in black Adidas jacket, reviewing papers; traffic cones and another masked person nearby"

from typing import Callable
search_images: Callable = tools['get_images_from_visual_query']
search_images

<function agentic_ai.tools.type.factory.ToolFactory._bind_tool.<locals>.async_wrapper(*args, **kwargs)>

In [14]:
result = await search_images(
    visual_query=VISUAL_QUERY,
    top_k=10,
    list_video_id=LIST_VIDEO_IDS,
    user_id=USER_ID
)

2025-11-04 10:48:24.773 | INFO     | agentic_ai.tools.clients.external.base:load_model:126 - service-image-embedding_model_loaded


1
512


In [15]:
import json
json.loads(result.text)

{'type': 'Search Image Results',
 'summary': 'Retrieved 10 visually similar frames across 1 videos.',
 'results': {'video1_111': [{'related_video_id': 'video1_111',
    'frame_index': 31802,
    'timestamp': '00:17:40.067',
    'caption_info': 'Bức ảnh chụp lại một khung cảnh diễn ra hoạt động kiểm soát hoặc làm việc trên đường phố, có sự tham gia của lực lượng Cảnh sát Giao thông (CSGT) Việt Nam và một số người dân.\n\n**Bối cảnh:** Sự việc dường như diễn ra ngoài trời, tại một khu vực có lề đường, với phông nền có các vật liệu xây dựng hoặc rào chắn tạm thời màu xanh dương và một vài vật cản giao thông hình nón màu cam. Có một bảng hiệu màu vàng hình tam giác cảnh báo mờ ảo phía sau. Trên góc trên bên phải có logo "ANTV HD", cho thấy đây là cảnh quay từ một bản tin truyền hình. Phía dưới màn hình có dòng chữ chạy (ticker) bằng tiếng Việt, xác nhận bối cảnh Việt Nam và một mốc thời gian là "18:16".\n\n**Nhân vật và Hành động:**\nCó ít nhất năm người xuất hiện, phần lớn là cán bộ CSGT.

In [16]:
search_images_caption: Callable = tools['get_images_from_caption_query']
search_images_caption

<function agentic_ai.tools.type.factory.ToolFactory._bind_tool.<locals>.async_wrapper(*args, **kwargs)>

In [17]:
CAPTION_QUERY = "Cảnh sát giao thông Việt Nam mặc đồng phục màu be, đội mũ kêpi viền đỏ, ngồi bên bàn ven đường cùng một người đàn ông mặc áo khoác Adidas đen, cùng xem giấy tờ; gần đó có người đeo khẩu trang và các cọc tiêu giao thông màu cam."
result_caption = await search_images_caption(
    caption_query=CAPTION_QUERY,
    top_k_each_request=50,
    top_k_final=30,
    weights=None,
    list_video_id=LIST_VIDEO_IDS,
    user_id=USER_ID
)

2025-11-04 10:48:24.946 | INFO     | agentic_ai.tools.clients.external.base:load_model:126 - service-text-embedding_model_loaded


In [18]:
CAPTION_QUERY = "Cảnh sát giao thông Việt Nam mặc đồng phục màu be, đội mũ kêpi viền đỏ, ngồi bên bàn ven đường cùng một người đàn ông mặc áo khoác Adidas đen, cùng xem giấy tờ; gần đó có người đeo khẩu trang và các cọc tiêu giao thông màu cam."
result_caption = await search_images_caption(
    caption_query=CAPTION_QUERY,
    top_k_each_request=50,
    top_k_final=30,
    weights=(0.7, 0.3),
    list_video_id=LIST_VIDEO_IDS,
    user_id=USER_ID
)

2025-11-04 10:48:24.979 | INFO     | agentic_ai.tools.clients.external.base:load_model:126 - service-text-embedding_model_loaded


In [19]:
get_segment_from_q_func: Callable = tools['get_segments_from_event_query']

result_segment = await get_segment_from_q_func(
    event_query=CAPTION_QUERY,
    top_k_each_request=50,
    top_k_final=30,
    weights=(0.7, 0.3),
    list_video_id=LIST_VIDEO_IDS,
    user_id=USER_ID
)


2025-11-04 10:48:25.015 | INFO     | agentic_ai.tools.clients.external.base:load_model:126 - service-text-embedding_model_loaded


In [20]:
multimodal_searching: Callable = tools['get_images_from_multimodal_query']

In [21]:
result_mul = await multimodal_searching(
    visual_query=VISUAL_QUERY,
    caption_query=CAPTION_QUERY,
    top_k_each_request=50,
    top_k_final=30,
    weights=(0.45, 0.45, 0.05),
    list_video_id=LIST_VIDEO_IDS,
    user_id=USER_ID
)

2025-11-04 10:48:25.043 | INFO     | agentic_ai.tools.clients.external.base:load_model:126 - service-image-embedding_model_loaded
2025-11-04 10:48:25.055 | INFO     | agentic_ai.tools.clients.external.base:load_model:126 - service-text-embedding_model_loaded


In [22]:
import json
result_mulll = json.loads(result_mul.text)
result_mulll

{'type': 'Search Image Results',
 'summary': 'Retrieved 30 visually similar frames across 2 videos.',
 'results': {'video1_111': [{'related_video_id': 'video1_111',
    'frame_index': 33370,
    'timestamp': '00:18:32.333',
    'caption_info': 'Bức ảnh chụp lại một cảnh kiểm tra giao thông vào ban đêm trên một con đường vắng vẻ, có thể là một chốt chặn được lập ra bởi lực lượng cảnh sát giao thông Việt Nam.\n\nỞ tiền cảnh bên trái, một cảnh sát giao thông đang đứng nghiêm trang, mặc đồng phục màu xanh lá cây và đội mũ kêpi, khoác ngoài là áo phản quang màu xanh chuối nổi bật. Anh ta đang cầm một cây gậy chỉ huy giao thông màu trắng đen ở tay phải, tay trái có vẻ đang cầm một vật nhỏ khác. Xung quanh anh ta có các cọc tiêu giao thông màu cam viền trắng được bố trí trên mặt đường để phân luồng hoặc đánh dấu khu vực kiểm tra.\n\nGần giữa bức ảnh, một cảnh sát khác mặc đồng phục màu vàng cát (có thể là cảnh sát cơ động hoặc cảnh sát giao thông làm nhiệm vụ đặc biệt) đang đứng thẳng, quan s

In [23]:
find_similar: Callable = tools['find_similar_images_from_image']

image_ref = ImageObjectInterface.model_validate(result_mulll['results']['video1_111'][0])

result_ref = await find_similar(
    reference_image=image_ref,
    top_k=10,
    list_video_id=LIST_VIDEO_IDS,
    user_id=USER_ID
)


2025-11-04 10:48:25.091 | INFO     | agentic_ai.tools.clients.external.base:load_model:126 - service-image-embedding_model_loaded
/home/tinhanhnguyen/miniconda3/lib/python3.12/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `list[str]` - serialized value may not be as expected [input_value='No semantic query relatd', input_type=str])
  return self.__pydantic_serializer__.to_python(


In [24]:
json.loads(result_ref.text)

{'type': 'Search Image Results',
 'summary': 'Retrieved 10 visually similar frames across 1 videos.',
 'results': {'video1_111': [{'related_video_id': 'video1_111',
    'frame_index': 33370,
    'timestamp': '00:18:32.333',
    'caption_info': 'Bức ảnh chụp lại một cảnh kiểm tra giao thông vào ban đêm trên một con đường vắng vẻ, có thể là một chốt chặn được lập ra bởi lực lượng cảnh sát giao thông Việt Nam.\n\nỞ tiền cảnh bên trái, một cảnh sát giao thông đang đứng nghiêm trang, mặc đồng phục màu xanh lá cây và đội mũ kêpi, khoác ngoài là áo phản quang màu xanh chuối nổi bật. Anh ta đang cầm một cây gậy chỉ huy giao thông màu trắng đen ở tay phải, tay trái có vẻ đang cầm một vật nhỏ khác. Xung quanh anh ta có các cọc tiêu giao thông màu cam viền trắng được bố trí trên mặt đường để phân luồng hoặc đánh dấu khu vực kiểm tra.\n\nGần giữa bức ảnh, một cảnh sát khác mặc đồng phục màu vàng cát (có thể là cảnh sát cơ động hoặc cảnh sát giao thông làm nhiệm vụ đặc biệt) đang đứng thẳng, quan s

In [25]:
res_seg = json.loads(result_segment.text)
res_seg

{'type': 'visual_segment_search_result',
 'summary': 'Retrieved 30 relevant segments across 2 videos based on semantic similarity to the query.',
 'results': {'video1_111': [{'related_video_id': 'video1_111',
    'start_frame_index': 31514,
    'end_frame_index': 31591,
    'start_time': '00:17:30.467',
    'end_time': '00:17:33.033',
    'caption_info': 'Trong một khung cảnh đường phố có vẻ mờ sương hoặc nhiều bụi, có thể là buổi sáng sớm, một sĩ quan cảnh sát giao thông Việt Nam với bộ đồng phục màu vàng kem và mũ lưỡi trai đỏ đang đứng tương tác với một người đi xe máy.\n\nNgười đi xe máy là một nam thanh niên đội mũ bảo hiểm màu đen, mặc áo khoác đen có viền lông cừu màu trắng ở cổ, đang ngồi trên xe máy (có thể là xe tay ga). Phía sau anh ta là một người phụ nữ đi cùng, đội mũ bảo hiểm màu xanh ngọc và quấn khăn che mặt màu hồng.\n\nCảnh sát giao thông đang cầm trên tay một tập tài liệu hoặc tờ rơi màu xanh nhạt (có hình ảnh phong cảnh, có thể liên quan đến an toàn giao thông hoặc

In [26]:
sample_segment_objects = []
for video_id, segments in res_seg['results'].items():
    for segment in segments:
            
        segment_obj = {
            'related_video_id': video_id,
            **segment
        }
        segobj = SegmentObjectInterface.model_validate(segment_obj)
        sample_segment_objects.append(segobj)
    

len(sample_segment_objects)

30

In [27]:
sample_image_interface = []
for video_id, images in json.loads(result_caption.text)['results'].items():
    for img in images:
        img_obj = {
            'related_video_id': video_id,
            ** img
        }
        ioj = ImageObjectInterface.model_validate(img_obj)
        sample_image_interface.append(ioj)

# Testing the utils tools

In [28]:
get_vid_from_seg: Callable = tools['get_video_from_segment']
video_retrieve = await get_vid_from_seg(sample_segment_objects[0])
video_retrieve

VideoInterface(video_id='video1_111', fps=30.0)

In [29]:
get_vid_from_img = tools['get_video_from_image']
video_retrieve = await get_vid_from_img(sample_image_interface[0])
video_retrieve

VideoInterface(video_id='video1_111', fps=30.0)

In [30]:
get_asr = tools['get_asr_from_video']
asr = await get_asr(video_retrieve)
asr

'Start time/index: 00:00:00.000/00000000\n        End time/index:   00:00:55.280/00001658\n        ASR content:      tin an ninh trật tự ngày hôm nay công an thành phố huế cho biết đã tạm giữ hình sự lê đức hạnh trú tại phường vĩ dạ để điều tra về hành vi chống người thi hành công vụ đối tượng đã đánh một chiến sĩ cảnh sát giao thông đang làm nhiệm vụ điều tiết xe cộ đi tránh lũ vào chiều ngày ba mươi mười tổ công tác tại đội hai phòng cảnh sát giao thông công an thành phố huế được phân công điều tiết giao thông để đảm bảo an toàn thông suốt tại khu vực cầu vĩ dạ lúc này lê đức hạnh điều khiển xe máy đi qua cầu đã không chấp hành hiệu lệnh của lực lượng thực hiện nhiệm vụ có lời lẽ chống đối xúc phạm rồi lạng lách bỏ chạy khi đi được khoảng ba mươi m hạnh bất ngờ dừng xe lại quay trở lại tấn công một chiến sĩ cảnh sát giao thông dùng tay chân đấm đá liên tục vào vùng mặt vào bụng lực lượng công an cùng với người dân có mặt đã nhanh chóng đưa hạnh về trụ sở công an phường vĩ dạ kết quả 

In [31]:
get_all_seg = tools['get_all_segment_info_from_video_interface']
segments = await get_all_seg(video_retrieve)
segments

[ArtifactMetadata(artifact_id='0b702c32a79a16d2265217023a65d2b38c0d98ecccbdf640c6774c6dac7f99363098aeb5dc5ee5fd955be45fea67f4c7280aeb3b318f520d9ee044ef8a9d774b', artifact_type='SegmentCaptionArtifact', user_id='anonymous', minio_url='s3://anonymous/caption/segment/video1_111/0_427_00:00:00.000_00:00:14.233.json', parent_artifact_id='3dc6a343a637497844223713a2d7708a68f82c1c62424bedda50d957e6f15b948aa3135528f51d91464d16f5fb32ef7987796d0efe2682c1aeacacbbd83a55de', created_at=datetime.datetime(2025, 11, 4, 2, 8, 34, 121591), artifact_metadata={}), ArtifactMetadata(artifact_id='ca1d753f386cf62c5824633321f91490a4d7e47b6187733f8bdd891b437b1965e1474cad29c83c3fd395cbe0e5438f382e250e2df287763c3a97d351ad1245ba', artifact_type='SegmentCaptionArtifact', user_id='anonymous', minio_url='s3://anonymous/caption/segment/video1_111/428_543_00:00:14.267_00:00:18.100.json', parent_artifact_id='3dc6a343a637497844223713a2d7708a68f82c1c62424bedda50d957e6f15b948aa3135528f51d91464d16f5fb32ef7987796d0efe2682c1ae

TextBlock(block_type='text', text='{\n  "type": "visual_segment_search_result",\n  "summary": "Retrieved 494 relevant segments across 1 videos based on semantic similarity to the query.",\n  "results": {\n    "video1_111": [\n      {\n        "related_video_id": "video1_111",\n        "start_frame_index": 0,\n        "end_frame_index": 427,\n        "start_time": "00:00:00.000",\n        "end_time": "00:00:14.233",\n        "caption_info": "Trong một cảnh quay từ chương trình tin tức của ANTV HD, một nữ phát thanh viên đang đứng ở phía bên trái màn hình, mặc đồng phục công an màu xanh rêu đậm với cầu vai có quân hàm và phù hiệu. Cô ấy đang cầm một tập tài liệu màu trắng trong tay và nhìn thẳng về phía trước với vẻ mặt nghiêm túc, chuẩn bị đưa tin.\\n\\nPhía sau nữ phát thanh viên là một màn hình lớn hiển thị một đoạn phóng sự hoặc cảnh quay khác, có hình ảnh một cuộc làm việc hoặc thẩm vấn. Trong hình ảnh trên màn hình, có hai người đàn ông đang ngồi đối diện nhau qua một chiếc bàn màu

In [32]:

sample_segment_objects[10]

SegmentObjectInterface(related_video_id='video1_111', start_frame_index=33113, end_frame_index=33210, start_time='00:18:23.767', end_time='00:18:27.000', caption_info='Dựa trên các hình ảnh được cung cấp, sự kiện chính diễn ra là một cuộc kiểm tra giao thông vào buổi tối.\n\n**Mô tả sự kiện:**\n\nTrong bối cảnh buổi tối, có thể là trên một con đường có đèn xe máy chiếu sáng, một cảnh sát giao thông Việt Nam đang tiến hành kiểm tra phương tiện. Cảnh sát mặc đồng phục màu vàng chanh có chữ "CGBT" (hoặc "CSGT") màu đỏ nổi bật trên lưng, đội mũ lưỡi trai màu trắng viền đỏ.\n\nHình ảnh tập trung vào cảnh sát đang đứng đối diện với một người đi xe máy. Người đi xe máy này có vẻ đang dừng lại để trình giấy tờ hoặc tương tác với cảnh sát; họ đang cầm một vật dụng nhỏ màu đen, có thể là điện thoại di động. Một người khác (có thể là người ngồi sau hoặc đi cùng) cũng có mặt gần đó. Một số người đi xe máy khác và ô tô đang di chuyển hoặc chờ đợi phía sau, với đèn pha xe máy và đèn xe ô tô chiếu sá

In [33]:
get_segs: Callable= tools['get_segments']
forward = await get_segs(
    current_segment=sample_segment_objects[10],
    hop=3,
    include_within_range=True,
    forward_or_backward='forward'
)

In [34]:
forward

[SegmentObjectInterface(related_video_id='video1_111', start_frame_index=33211, end_frame_index=33305, start_time='00:18:27.033', end_time='00:18:30.167', caption_info='Trong đoạn video được quay vào buổi tối, tại một khu vực có vẻ là ven đường, một cảnh sát giao thông (CSGT) Việt Nam đang tiến hành kiểm tra người điều khiển xe máy.\n\nCảnh sát mặc đồng phục màu vàng cát, đội mũ lưỡi trai, và khoác áo phản quang màu vàng có chữ "C.S.G.T". Anh ta đang cầm một thiết bị kiểm tra (có thể là máy đo nồng độ cồn) và đưa về phía người điều khiển xe máy.\n\nNgười điều khiển xe máy, mặc áo khoác màu xanh dương và xám, đang ngồi trên xe và đội mũ bảo hiểm có họa tiết màu đỏ, trắng, đen. Người này đang dùng tay chạm vào mũ bảo hiểm của mình, có vẻ như đang điều chỉnh hoặc chuẩn bị tháo mũ theo yêu cầu của cảnh sát.\n\nCả hai người đang đối diện nhau. Cảnh sát nhìn thẳng về phía người vi phạm, trong khi người đội mũ bảo hiểm hướng mặt về phía cảnh sát.\n\nDưới góc màn hình có dòng chữ chạy của ANTV

In [35]:
# get_img: Callable = tools['get_images']
# images = await get_img(
#     image=sample_image_interface[9],
#     hop=3,
#     include_within_range=True,
#     forward_or_backward='backward'
# )

In [36]:
images

[{'related_video_id': 'video2_222',
  'frame_index': 5321,
  'timestamp': '00:02:57.367',
  'caption_info': 'Trong một khung hình đậm chất hành động, một quân nhân đang ở tư thế sẵn sàng chiến đấu trong một khu vực có vẻ là rừng hoặc bìa rừng với nhiều thân cây cao vút phía sau dưới ánh sáng ban ngày khá sáng.\n\nQuân nhân này mặc quân phục rằn ri màu xanh lá cây và nâu, đội mũ trùm đầu (có thể là balaclava hoặc mũ đội dưới mũ bảo hiểm) cùng màu, và đeo miếng đệm bảo vệ khuỷu tay lớn màu nâu nhạt che phủ rất rõ ràng ở cánh tay trái. Người lính đang cúi thấp người, dường như đang nấp sau một ụ đất hoặc công sự thấp ở phía trước, mắt chăm chú nhìn qua ống ngắm của khẩu súng trường tấn công.\n\nKhẩu súng mà người lính đang giữ là một loại súng trường có băng đạn cong, trông giống như một biến thể của AK (có thể là AK-74 hoặc một mẫu tương tự), được nạp đạn và người lính đang đặt má sát vào báng súng, sẵn sàng khai hỏa. Bàn tay trái của anh ta nắm chắc phần ốp lót tay phía trước.\n\nBối cả

In [37]:
sample_image_interface[10]

ImageObjectInterface(related_video_id='video1_111', frame_index=29633, timestamp='00:16:27.767', caption_info='Bức ảnh chụp cận cảnh hai cán bộ cảnh sát giao thông Việt Nam đang làm việc ngoài trời, có thể là tại hiện trường một vụ việc hoặc một đợt kiểm tra.\n\nCả hai sĩ quan đều mặc quân phục màu nâu vàng đặc trưng của lực lượng Cảnh sát Giao thông (CSGT), đội mũ tai bèo màu vàng nhạt và đeo khẩu trang y tế, cho thấy bối cảnh có thể là trong thời gian dịch bệnh hoặc vì lý do an toàn khi làm nhiệm vụ. Trên vai áo của họ có cầu vai màu đỏ có sọc vàng, và nổi bật là logo hình khiên của lực lượng CSGT được gắn ở cánh tay.\n\nSĩ quan bên trái đang cúi xuống, tay dường như đang trao đổi hoặc kiểm tra giấy tờ với một người khác không xuất hiện rõ trong khung hình (chỉ thấy một phần cánh tay). Anh ta mặc thêm áo phản quang màu vàng chanh bên ngoài đồng phục.\n\nSĩ quan bên phải cũng đang cúi người xuống để ghi chép hoặc xem xét tài liệu trên một bề mặt thấp (có thể là bàn tạm hoặc sổ ghi ché

In [38]:
'00:18:50.200'

'00:18:50.200'

In [39]:
extract_f = tools['extract_frame_time']
extract_obj_interface = await extract_f(
    video_interface=video_retrieve,
    timestamp='00:18:50.200',
    agent_bucket='test-agent',
    agent_object_folder='test1'
)

In [40]:
extract_obj_interface

ImageObjectInterface(related_video_id='video1_111', frame_index=33906, timestamp='00:18:50.200', caption_info=None, minio_path='s3://test-agent/test1/video1_111/frames/33906.webp', score='No semantic score', query='No semantic query relatd', reference_query_image=None)

In [41]:
extract_f2: Callable = tools['extract_frames_by_time_window']
extract_f2_obj = await extract_f2(
    video_interface=video_retrieve,
    start_time='00:18:50.200',
    end_time='00:20:00.000',
    fps_sample_rate=60,
    agent_bucket='test-agent',
    agent_object_folder='test1'
)

In [42]:
json.loads(extract_f2_obj.text)

{'type': 'Search Image Results',
 'summary': 'Retrieved 35 visually similar frames across 1 videos.',
 'results': {'video1_111': [{'related_video_id': 'video1_111',
    'frame_index': 33906,
    'timestamp': '00:18:50.200',
    'caption_info': None,
    'minio_path': 's3://test-agent/test1/video1_111/frames/33906.webp',
    'score': 'No semantic score',
    'query': 'No semantic query relatd',
    'reference_query_image': None},
   {'related_video_id': 'video1_111',
    'frame_index': 33966,
    'timestamp': '00:18:52.200',
    'caption_info': None,
    'minio_path': 's3://test-agent/test1/video1_111/frames/33966.webp',
    'score': 'No semantic score',
    'query': 'No semantic query relatd',
    'reference_query_image': None},
   {'related_video_id': 'video1_111',
    'frame_index': 34026,
    'timestamp': '00:18:54.200',
    'caption_info': None,
    'minio_path': 's3://test-agent/test1/video1_111/frames/34026.webp',
    'score': 'No semantic score',
    'query': 'No semantic query 

In [43]:
read_img = tools['read_image']
img_b = read_img(sample_image_interface[0])

s3://anonymous/images/video1_111/00033370_00:18:32.333.webp
anonymous


In [44]:
# read_segment = tools['read_segment']
# segment_b = await read_segment(sample_segment_objects[0])

In [45]:
get_asr_fi = tools['get_related_asr_from_image']

asrrrr = await get_asr_fi(sample_image_interface[0], window_seconds=30.0)

[TimestampedToken(text='tin an ninh trật tự ngày hôm nay công an thành phố huế cho biết đã tạm giữ hình sự lê đức hạnh trú tại phường vĩ dạ để điều tra về hành vi chống người thi hành công vụ đối tượng đã đánh một chiến sĩ cảnh sát giao thông đang làm nhiệm vụ điều tiết xe cộ đi tránh lũ vào chiều ngày ba mươi mười tổ công tác tại đội hai phòng cảnh sát giao thông công an thành phố huế được phân công điều tiết giao thông để đảm bảo an toàn thông suốt tại khu vực cầu vĩ dạ lúc này lê đức hạnh điều khiển xe máy đi qua cầu đã không chấp hành hiệu lệnh của lực lượng thực hiện nhiệm vụ có lời lẽ chống đối xúc phạm rồi lạng lách bỏ chạy khi đi được khoảng ba mươi m hạnh bất ngờ dừng xe lại quay trở lại tấn công một chiến sĩ cảnh sát giao thông dùng tay chân đấm đá liên tục vào vùng mặt vào bụng lực lượng công an cùng với người dân có mặt đã nhanh chóng đưa hạnh về trụ sở công an phường vĩ dạ kết quả kiểm tra cho thấy hạnh dương tính với ma túy và cần sa đối tượng có nồng độ cồn là không chín

In [46]:
get_asr_fs = tools['get_related_asr_from_segment']
asr_seg =  await get_asr_fs(sample_segment_objects[0], window_seconds=30.0)

In [47]:
asr_seg

'\n    ASR transcript context around the segment:\n\n    ▶ Segment range: 00:17:30.467 → 00:17:33.033\n    ▶ Frame range: 31514 → 31591\n    ▶ Context window: ±30.0 seconds\n\n    --------------------- TRANSCRIPT CONTEXT ---------------------\n    Start time/index: 00:16:08.480/00029054\n        End time/index:   00:18:21.200/00033036\n        ASR content:      chỉ đạo của cục cảnh sát giao thông bộ công an về mở đợt cao điểm kiểm tra xử lý vi phạm nồng độ cồn trên phạm vi cả nước từ mười hai giờ ngày mười tám tháng mười năm hai nghìn lẻ hai mươi lăm lực lượng cảnh sát giao thông công an tỉnh thái nguyên đã đồng loạt ra quân kiểm tra xử lý vi phạm trên toàn địa bàn các tổ công tác được bố trí tại nhiều địa bàn trọng điểm như khu vực nhà hàng quán ăn quán bar khu vui chơi giải trí khu công nghiệp quốc lộ và các tuyến đường nội thị tất cả các trường hợp vi phạm liên quan đến nồng độ cồn đều được ghi hình lập biên bản tạm giữ phương tiện giấy phép lái xe và xử lý nghiêm theo quy định tron

In [48]:
sample_segment_objects[0]

SegmentObjectInterface(related_video_id='video1_111', start_frame_index=31514, end_frame_index=31591, start_time='00:17:30.467', end_time='00:17:33.033', caption_info='Trong một khung cảnh đường phố có vẻ mờ sương hoặc nhiều bụi, có thể là buổi sáng sớm, một sĩ quan cảnh sát giao thông Việt Nam với bộ đồng phục màu vàng kem và mũ lưỡi trai đỏ đang đứng tương tác với một người đi xe máy.\n\nNgười đi xe máy là một nam thanh niên đội mũ bảo hiểm màu đen, mặc áo khoác đen có viền lông cừu màu trắng ở cổ, đang ngồi trên xe máy (có thể là xe tay ga). Phía sau anh ta là một người phụ nữ đi cùng, đội mũ bảo hiểm màu xanh ngọc và quấn khăn che mặt màu hồng.\n\nCảnh sát giao thông đang cầm trên tay một tập tài liệu hoặc tờ rơi màu xanh nhạt (có hình ảnh phong cảnh, có thể liên quan đến an toàn giao thông hoặc tuyên truyền) và đưa cho người đi xe máy xem hoặc phát cho anh ta. Người đàn ông đang chăm chú nhìn vào tờ rơi này. Một sĩ quan cảnh sát khác, cũng mặc đồng phục và đội mũ, một phần cơ thể 

In [49]:
ftt = tools['frame_to_timecode']
timecode=  ftt(
    frame_index=10000,
    fps=25
)

In [50]:
timecode

'00:06:40.000'

In [52]:
llm_enhance_tools = tools['enhance_visual_query']

result_enhanced_query = await llm_enhance_tools(
    raw_query="Hãy tìm cho tôi khoảnh khắc của một con chó đang chạy nhảy trên cỏ",
    variants=['Hãy nâng cấp không gian', 'làm rõ hơn về ngữ cảnh', 'Thêm một số tình tiết liên quan']
)

/tmp/ipykernel_251850/4099823706.py:3: RuntimeWarning: coroutine 'ToolFactory._bind_tool.<locals>.async_wrapper' was never awaited
  result_enhanced_query = await llm_enhance_tools(


In [53]:
result_enhanced_query

['A photo of a happy dog running and jumping on green grass under bright sunlight.',
 'A dynamic shot of an energetic dog leaping through lush green grass, full of joy.',
 'A wide-angle picture of a playful dog bounding across a vibrant grassy field at sunset.',
 "A close-up image of a furry dog's paws kicking up dew-kissed grass while running.",
 'A heartwarming scene of a joyful dog freely running and jumping in a sunlit meadow.']